# Tutorial 9: Audio Generation with IndexTTS

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/01_audio/9.IndexTTS.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

## Getting started

### Environment Setup

Let's first check if our environment is all set up.

In [1]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython
from IPython.display import display

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")
    !sudo apt-get install sox ffmpeg
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %pip install git+https://github.com/index-tts/index-tts.git
    %hf download "sdialog/voices-jsalt" --repo-type dataset
    %hf download "facebook/w2v-bert-2.0" --repo-type model
    # %hf download "sdialog/voices-libritts" --repo-type dataset
    %cd ..
else:
    print("Running in Jupyter Notebook")

Running in Jupyter Notebook


### Load the example dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [2]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()
original_dialog.turns = original_dialog.turns[0:4]
print(len(original_dialog.turns))

/Users/yanislabrak/Desktop/HUB/PostJSALT/sdialog/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[dialog_id] 86ff0e9e-4b2e-4379-87d9-8c6000424b4f
[model] {'name': 'amazon:anthropic.claude-3-5-sonnet-20240620-v1:0', 'temperature': 0.7, 'max_tokens': 512, 'region_name': 'us-east-1'}
[seed] 42
--- Dialogue Begins ---
[Marie] Hello doctor, I've been having headaches for two weeks and I'm very tired.
[Dr. Martin] I'm sorry to hear that donald. Can you tell me where the pain is strongest?
[Marie] Mostly at the front of my head and behind my eyes.
[Dr. Martin] I see. Have you been under a lot of stress lately?
[Marie] Yes, work has been stressful and my sleep hasn't been great.
[Dr. Martin] That could explain the tension. I'd like to check your blood pressure now.
[Marie] Of course, go ahead.
[Dr. Martin] Your blood pressure looks normal, but your neck muscles are very tense.
[Marie] So it might just be stress?
[Dr. Martin] Most likely. I recommend stretching, better sleep habits, and keeping a headache diary.
[Marie] Thank you, doctor. I'll start doing that.
--- Dialogue Ends ---
4


In [3]:
INDEXTTS_VERSION = "1.5"
# INDEXTTS_VERSION = "2"

In [4]:
from sdialog.audio.normalizers import LowercaseNormalizer, WhisperNormalizer, ReplaceCommaWithDotNormalizer
normalizers = [ReplaceCommaWithDotNormalizer(), LowercaseNormalizer(), WhisperNormalizer()]

In [5]:
from sdialog.audio.tts import IndexTTS
from sdialog.audio.voice_database import HuggingfaceVoiceDatabase

if INDEXTTS_VERSION == "1.5":
    tts_engine = IndexTTS(model_dir="checkpoints-15", cfg_path="checkpoints-15/config.yaml", version="1", text_normalizers=normalizers)
else:
    tts_engine = IndexTTS(model_dir="checkpoints", cfg_path="checkpoints/config.yaml", version="2", text_normalizers=normalizers)

[2025-12-06 20:16:58] WARNING:indextts.gpt.transformers_modeling_utils:GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


>> GPT weights restored from: checkpoints-15/gpt.pth
Removing weight norm...
>> bigvgan weights restored from: checkpoints-15/bigvgan_generator.pth
>> TextNormalizer loaded
>> bpe model loaded from: checkpoints-15/bpe.model


In [ ]:
voice_database = HuggingfaceVoiceDatabase("sdialog/voices-celebrities")
# voice_database = HuggingfaceVoiceDatabase("sdialog/voices-jsalt")
# voice_database = HuggingfaceVoiceDatabase("sdialog/voices-libritts")

Using the latest cached version of the dataset since sdialog/voices-jsalt couldn't be found on the Hugging Face Hub
[2025-12-06 20:17:00] WARNING:datasets.load:Using the latest cached version of the dataset since sdialog/voices-jsalt couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/yanislabrak/.cache/huggingface/datasets/sdialog___voices-jsalt/default/0.0.0/c372daa7f3e88c6f68e95cedf38c9c6fcd271378 (last modified on Sat Nov 29 22:11:08 2025).
[2025-12-06 20:17:00] WARNING:datasets.packaged_modules.cache.cache:Found the latest cached dataset configuration 'default' at /Users/yanislabrak/.cache/huggingface/datasets/sdialog___voices-jsalt/default/0.0.0/c372daa7f3e88c6f68e95cedf38c9c6fcd271378 (last modified on Sat Nov 29 22:11:08 2025).
[2025-12-06 20:17:01] INFO:sdialog.audio:[Voice Database] Has been populated with 5 voices


## Setup stage: Audio Dialog and Audio Pipeline

In [7]:
from sdialog.audio.dialog import AudioDialog
dialog: AudioDialog = AudioDialog.from_dialog(original_dialog)

In [8]:
from sdialog.audio.utils import SourceVolume, SourceType
from sdialog.audio.room import SpeakerSide, Role, RoomPosition
from sdialog.audio.jsalt import MedicalRoomGenerator, RoomRole

In [9]:
room = MedicalRoomGenerator().generate(args={"room_type": RoomRole.TREATMENT})
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_1, furniture_name="desk", max_distance=1.0, side=SpeakerSide.FRONT)
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_2, furniture_name="desk", max_distance=1.0, side=SpeakerSide.BACK)

In [ ]:
new_audio_dialog = original_dialog.to_audio(
    tts_engine=tts_engine,
    voice_database=voice_database,
    perform_room_acoustics=True,
    room=room,
    dir_audio="./outputs_indextts",
    dialog_dir_name="demo_indextts-15" if INDEXTTS_VERSION == "1.5" else "demo_indextts",
    room_name=f"base_room",
    background_effect="white_noise",
    foreground_effect="ac_noise_minimal",
    foreground_effect_position=RoomPosition.TOP_RIGHT,
    source_volumes={
        SourceType.ROOM: SourceVolume.HIGH,
        SourceType.BACKGROUND: SourceVolume.VERY_LOW
    },
    kwargs_pyroom={
        "ray_tracing": True,
        "air_absorption": True
    },
    override_tts_audio=True,
    verbose=True,
    voices={
        Role.SPEAKER_1: ("donald_trump","english"),
        Role.SPEAKER_2: ("bill_gates","english"),
    }
)

[2025-12-06 20:17:07] INFO:sdialog.audio:[Initialization] Audio file format for generation is set to wav
[2025-12-06 20:17:07] INFO:sdialog.audio:[Initialization] No existing file found for the dialogue (86ff0e9e-4b2e-4379-87d9-8c6000424b4f), starting from scratch...
[2025-12-06 20:17:07] INFO:sdialog.audio:[Step 1] Generating audio recordings from the utterances of the dialogue: 86ff0e9e-4b2e-4379-87d9-8c6000424b4f
Generating utterances audios:   0%|          | 0/4 [00:00<?, ?it/s]

>> starting inference...


Generating utterances audios:  25%|██▌       | 1/4 [01:07<03:23, 68.00s/it]

>> Reference audio length: 18.99 seconds
>> gpt_gen_time: 61.21 seconds
>> gpt_forward_time: 0.57 seconds
>> bigvgan_time: 6.19 seconds
>> Total inference time: 67.99 seconds
>> Generated audio length: 4.95 seconds
>> RTF: 13.7374
>> starting inference...


Generating utterances audios:  50%|█████     | 2/4 [02:09<02:08, 64.09s/it]

...

Generating utterances audios:  75%|███████▌  | 3/4 [02:43<00:50, 50.57s/it]

>> Reference audio length: 18.99 seconds
>> gpt_gen_time: 30.25 seconds
>> gpt_forward_time: 0.58 seconds
>> bigvgan_time: 3.60 seconds
>> Total inference time: 34.47 seconds
>> Generated audio length: 3.50 seconds
>> RTF: 9.8517
>> starting inference...


Generating utterances audios: 100%|██████████| 4/4 [03:30<00:00, 52.59s/it]
[2025-12-06 20:20:38] INFO:sdialog.audio:[Step 1] Audio files have been saved here: ./outputs_indextts/demo_indextts-15/exported_audios/audio_pipeline_step1.wav
[2025-12-06 20:20:38] INFO:sdialog.audio:[Step 2] Sending utterances to dSCAPER...
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] ==============================
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] # Audio sent to dSCAPER
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] ==============================
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] Already present: 0
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] Correctly added: 4
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] Errors: 0
[2025-12-06 20:20:38] INFO:sdialog.audio:[dSCAPER] ==============================
[2025-12-06 20:20:38] INFO:sdialog.audio:[Step 2] Generating timeline from dSCAPER...


>> Reference audio length: 65.44 seconds
>> gpt_gen_time: 39.23 seconds
>> gpt_forward_time: 2.59 seconds
>> bigvgan_time: 4.55 seconds
>> Total inference time: 46.51 seconds
>> Generated audio length: 4.35 seconds
>> RTF: 10.6870


ValueError: operands could not be broadcast together with shapes (746993,1) (746995,1) 

In [ ]:
new_audio_dialog.display()